# Phase 2-ZK: λ_d + ZeckendorfReadout

Replaces lm_head (44.8M params) with ZeckendorfReadout (86K).
Continues from Phase 2 checkpoint at step 25000.

**Architecture:** LDStack 12×D=896 + Zeckendorf tree (K=24) + adaptive depth + learnable V
**Loss:** NLL over Zeckendorf tree (per-target, O(K) per token)

In [ ]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/eva_lambda'
os.makedirs(ROOT, exist_ok=True)

# List what's available
print('Files on Drive:')
for f in os.listdir(ROOT):
    fp = os.path.join(ROOT, f)
    sz = os.path.getsize(fp) / 1e9
    print(f'  {f}: {sz:.2f} GB')

In [ ]:
# --- Clone repo ---
%cd /content
if not os.path.exists('EVA-Ai'):
    !git clone https://github.com/anomalyco/EVA-Ai.git
%cd EVA-Ai
!git pull

In [ ]:
# --- Restore data + checkpoint from Drive ---
import shutil

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('checkpoints_zk', exist_ok=True)

# Search Drive for the files (may have different names)
drive_files = os.listdir(ROOT)

data_src = None
ckpt_src = None
for f in drive_files:
    if 'russian_chunks' in f and f.endswith('.npy'):
        data_src = f
    if 'model_step' in f and f.endswith('.pt'):
        ckpt_src = f

if data_src:
    print(f'Copying data: {data_src}')
    shutil.copy2(os.path.join(ROOT, data_src), 'russian_chunks.npy')
else:
    raise FileNotFoundError(f'russian_chunks.npy not found in {ROOT}. '
                            f'Upload it first to Google Drive at {ROOT}/')

if ckpt_src:
    print(f'Copying checkpoint: {ckpt_src}')
    shutil.copy2(os.path.join(ROOT, ckpt_src), 'checkpoints/model_step25000.pt')
else:
    print('WARNING: No checkpoint found. Training from scratch.')

# Verify
print()
for fn in ['russian_chunks.npy', 'checkpoints/model_step25000.pt']:
    if os.path.exists(fn):
        size = os.path.getsize(fn) / 1e9
        print(f'  {fn}: {size:.2f} GB')

In [ ]:
# ─── Install deps ────────────────────────────────────────────────────
!pip install -q numpy torch

import torch, numpy as np
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ─── Training ─────────────────────────────────────────────────────────
import sys, math, time
sys.path.insert(0, '.')

# Check data
arr = np.load('russian_chunks.npy')
print(f'Data: {arr.shape[0]:,} chunks, {arr.shape[1]} tok each')

RESUME_FROM = 'checkpoints/model_step25000.pt'
if not os.path.exists(RESUME_FROM):
    print(f'WARNING: {RESUME_FROM} not found, training from scratch!')
    RESUME_FROM = None

!python train_phase2_zk.py \
    --data russian \
    --train_chunks 10000 \
    --eval_chunks 500 \
    --epochs 3 \
    {f'--resume {RESUME_FROM}' if RESUME_FROM else ''}

In [ ]:
# ─── Backup checkpoints to Drive ──────────────────────────────────────
import glob, shutil

for fn in glob.glob('checkpoints_zk/*.pt'):
    dest = os.path.join(ROOT, os.path.basename(fn))
    shutil.copy2(fn, dest)
    print(f'Copied {fn} -> Drive')

# Free up space: keep only last 3 + best
ckpts = sorted(glob.glob('checkpoints_zk/phase2zk_epoch*.pt'))
for fn in ckpts[:-3]:
    os.remove(fn)
    print(f'Removed old: {fn}')

In [ ]:
# ─── Quick comparison: Zeckendorf vs lm_head ─────────────────────────
import torch, numpy as np, math
from ld_model.core import LDConfig, LDStack
from ld_model.readout import ZeckendorfReadout

DEVICE = 'cuda'; D=896; VOCAB=50000
cfg = LDConfig(); cfg.D=D; cfg.n_layers=12; cfg.n_modes=4; cfg.vocab=VOCAB
cfg.bottleneck=256; cfg.adaptive_depth=True; cfg.learnable_V=True; cfg.V_rank=8

# Load Phase2 model with lm_head
print('Loading Phase2 (lm_head)...')
orig = torch.load('checkpoints/model_step25000.pt', map_location=DEVICE, weights_only=True)
sd_orig = orig['model_fp16'] if 'model_fp16' in orig else orig['model_state_dict']

class Phase2Model(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = torch.nn.Embedding(VOCAB, D)
        self.stack = LDStack(cfg)
        self.lm_head = torch.nn.Linear(D, VOCAB, bias=False)
    def forward(self, x):
        return self.lm_head(self.stack(self.embed(x)))

model_lm = Phase2Model().to(DEVICE)
sd_lm = {k: v.float() if v.dtype==torch.float16 else v for k,v in sd_orig.items()}
model_lm.load_state_dict({k:v for k,v in sd_lm.items() if k in model_lm.state_dict()}, strict=False)
model_lm.eval()

# Load Phase2-ZK
print('Loading Phase2-ZK...')
model_zk = Phase2ZKModel().to(DEVICE)
best_zk = sorted(glob.glob('checkpoints_zk/phase2zk_*.pt'))
if best_zk:
    sd_z = torch.load(best_zk[-1], map_location=DEVICE, weights_only=True)['model_state_dict']
    model_zk.load_state_dict(sd_z, strict=False)
model_zk.eval()

# Compare
arr = np.load('russian_chunks.npy')
loss_lm, loss_zk = 0.0, 0.0
for _ in range(50):
    idx = np.random.randint(0, len(arr), 4)
    x = torch.from_numpy(arr[idx, :128].copy()).long().to(DEVICE)
    with torch.no_grad():
        logits = model_lm(x)
        loss_lm += torch.nn.functional.cross_entropy(logits.reshape(-1, VOCAB), x.reshape(-1)).item()
        h = model_zk.stack(model_zk.embed(x))
        log_p = model_zk.readout.log_probs_for_target(h.reshape(-1, D), x.reshape(-1))
        loss_zk += -log_p.mean().item()

loss_lm /= 50; loss_zk /= 50
print(f'\n{"="*50}')
print(f'{"Model":<25} {"Loss":<10} {"PPL":<10}')
print(f'{"-"*45}')
print(f'{"lm_head (44.8M)":<25} {loss_lm:<10.4f} {math.exp(loss_lm):<10.1f}')
print(f'{"Zeckendorf (86K)":<25} {loss_zk:<10.4f} {math.exp(loss_zk):<10.1f}')
print(f'{"Ratio":<25} {"":>10} {math.exp(loss_lm)/math.exp(loss_zk):>10.2f}x')
print(f'{"="*50}')